# 08 — Robustness Evaluation — **UPDATED**

**Changes:** Loads `label_encoders.json` from NB05 (consistent 9-class mapping).
Run **after NB05 and NB06**.


In [ ]:
import os, json, glob, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics import (f1_score, accuracy_score, matthews_corrcoef,
                              roc_auc_score, classification_report)

warnings.filterwarnings("ignore")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")


## Config — must match NB05

In [ ]:
MODELS_DIR = "../outputs/models_v2_fix"
SPLITS_DIR = "../data/splits"

CONFIG = {
    "models": {
        "banglabert": "csebuetnlp/banglabert",
        "muril":      "google/muril-base-cased",
        "xlmr":       "xlm-roberta-base",
    },
    "max_length":      128,
    "batch_size":       64,    # inference-only — can be larger than training
    "eval_batch_size":  64,
}


## Load global label encoders from NB05

In [ ]:
LE_PATH = f"{MODELS_DIR}/label_encoders.json"
assert os.path.exists(LE_PATH), (
    f"label_encoders.json not found at {LE_PATH}. Run NB05 first."
)
with open(LE_PATH) as _f:
    _raw_le = json.load(_f)

label_encoders = {}
for task, enc in _raw_le.items():
    if task == "binary":
        label_encoders[task] = {int(k): v for k, v in enc.items()}
    else:
        label_encoders[task] = enc

TASK_CONFIG = {
    "binary": {
        "col": "label_binary",
        "num_classes": len(label_encoders["binary"]),
    },
    "abuse_type": {
        "col": "label_type",
        "num_classes": len(label_encoders["abuse_type"]),
    },
}

print(f"binary    classes: {TASK_CONFIG['binary']['num_classes']}")
print(f"abuse_type classes: {TASK_CONFIG['abuse_type']['num_classes']}")
print(f"abuse_type mapping: {label_encoders['abuse_type']}")


## Apply same label consolidation as NB05

In [ ]:
TYPE_MAP = {
    "none":"none","not bully":"none",
    "threat":"threat","threat,spam":"threat",
    "callToViolence":"threat","callToViolence_slander":"threat",
    "callToViolence_gender":"threat","callToViolence_religion":"threat",
    "callToViolence_religion_slander":"threat",
    "callToViolence_gender_religion_slander":"threat",
    "callToViolence_gender_slander":"threat",
    "religious,threat":"threat","sexual,threat":"threat","sexual,religious,threat":"threat",
    "sexual":"sexual","sexual,religious":"sexual","sexual,spam":"sexual",
    "religious":"religious","Religious":"religious","religion":"religious",
    "religious,spam":"religious","religion_slander":"religious",
    "gender_religion":"religious","gender_religion_slander":"religious",
    "gender":"gender","Gender":"gender","gender_slander":"gender",
    "Political":"political",
    "Personal Offense":"personal","Body Shaming":"personal",
    "Origin":"personal","slander":"personal","Misc":"personal",
    "Abusive/Violence":"abusive","troll":"abusive",
    "spam":"other",
}
PRIORITY = ["threat","sexual","religious","gender","political","abusive","personal","other","none"]

def consolidate_type(val):
    if not isinstance(val, str) or not val.strip(): return "none"
    val = val.strip()
    if val in TYPE_MAP: return TYPE_MAP[val]
    parts = [p.strip() for p in val.replace(";",",").split(",")]
    cands = [TYPE_MAP[p] for p in parts if p in TYPE_MAP]
    if cands:
        for pc in PRIORITY:
            if pc in cands: return pc
    sl = val.lower().replace("_"," ")
    for kw, cls in [("threat","threat"),("calltoviolence","threat"),("sexual","sexual"),
                    ("religious","religious"),("religion","religious"),("gender","gender"),
                    ("political","political"),("abusive","abusive"),("violence","abusive"),
                    ("personal","personal"),("slander","personal"),("origin","personal"),
                    ("body","personal"),("misc","personal"),("spam","other")]:
        if kw in sl.replace(" ",""): return cls
    return "other"

# Load and consolidate all splits
train_df = pd.read_csv(f"{SPLITS_DIR}/random_train.csv")
val_df   = pd.read_csv(f"{SPLITS_DIR}/random_val.csv")
test_df  = pd.read_csv(f"{SPLITS_DIR}/random_test.csv")
for df_ in [train_df, val_df, test_df]:
    df_["label_type"] = df_["label_type"].apply(consolidate_type)

TEXT_COL = "text_clean" if "text_clean" in train_df.columns else "text"
print(f"Text col: {TEXT_COL}   |   abuse_type classes in train: {train_df['label_type'].nunique()}")


## Model architecture — must match NB05 exactly

In [ ]:
class TaskHead(nn.Module):
    def __init__(self, hidden, n_cls, dropout=0.25, inner=384):
        super().__init__()
        inner = min(inner, hidden)
        self.net = nn.Sequential(
            nn.Dropout(dropout), nn.Linear(hidden, inner),
            nn.GELU(), nn.LayerNorm(inner),
            nn.Dropout(dropout), nn.Linear(inner, n_cls),
        )
    def forward(self, x): return self.net(x)


class MultiTaskTransformer(nn.Module):
    def __init__(self, model_name, task_config, dropout=0.25, head_hidden_dim=384):
        super().__init__()
        self.encoder  = AutoModel.from_pretrained(model_name)
        hidden        = self.encoder.config.hidden_size
        mtype         = self.encoder.config.model_type.lower()
        self._use_tti = mtype not in ("roberta","xlm-roberta")
        self.heads    = nn.ModuleDict({
            t: TaskHead(hidden, cfg["num_classes"], dropout, head_hidden_dim)
            for t, cfg in task_config.items()
        })

    def _pool(self, out, mask):
        hs   = out.last_hidden_state
        cls  = hs[:, 0]
        mean = (hs * mask.unsqueeze(-1).float()).sum(1) / mask.sum(1, keepdim=True).float().clamp(1)
        return 0.5 * cls + 0.5 * mean

    def forward(self, input_ids, attention_mask, token_type_ids=None):
        kw = {"input_ids": input_ids, "attention_mask": attention_mask}
        if self._use_tti and token_type_ids is not None: kw["token_type_ids"] = token_type_ids
        out = self.encoder(**kw)
        p   = self._pool(out, attention_mask)
        return {t: h(p) for t, h in self.heads.items()}


class RobustnessDataset(Dataset):
    def __init__(self, df, tokenizer, max_length, text_col):
        self.texts = df[text_col].fillna("").astype(str).tolist()
        self.tok   = tokenizer
        self.ml    = max_length
        binary_enc  = label_encoders["binary"]
        abuse_enc   = label_encoders["abuse_type"]
        self.binary = [int(binary_enc.get(int(v) if not isinstance(v,str) else v,-1))
                        for v in df["label_binary"]]
        self.abuse  = [int(abuse_enc.get(str(v),-1)) for v in df["label_type"]]

    def __len__(self): return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tok(self.texts[idx], max_length=self.ml, truncation=True, padding=False)
        item = {k: torch.tensor(v, dtype=torch.long) for k, v in enc.items()}
        item["label_binary"]     = torch.tensor(self.binary[idx], dtype=torch.long)
        item["label_abuse_type"] = torch.tensor(self.abuse[idx],  dtype=torch.long)
        return item


def collate_rob(features):
    lkeys  = [k for k in features[0] if k.startswith("label_")]
    labels = {k: torch.stack([f[k] for f in features]) for k in lkeys}
    tfeats = [{k: v for k, v in f.items() if not k.startswith("label_")} for f in features]
    from transformers import AutoTokenizer   # reuse pad
    # manual pad
    max_l = max(len(f["input_ids"]) for f in tfeats)
    ids  = torch.zeros(len(tfeats), max_l, dtype=torch.long)
    msk  = torch.zeros(len(tfeats), max_l, dtype=torch.long)
    has_tti = "token_type_ids" in tfeats[0]
    tti  = torch.zeros(len(tfeats), max_l, dtype=torch.long) if has_tti else None
    for i, f in enumerate(tfeats):
        L = len(f["input_ids"])
        ids[i,:L] = torch.tensor(f["input_ids"], dtype=torch.long)
        msk[i,:L] = torch.tensor(f["attention_mask"], dtype=torch.long)
        if has_tti: tti[i,:L] = torch.tensor(f["token_type_ids"], dtype=torch.long)
    batch = {"input_ids": ids, "attention_mask": msk}
    if has_tti: batch["token_type_ids"] = tti
    batch.update(labels)
    return batch

print("Robustness model + dataset defined ✅")


## Load all saved checkpoints

In [ ]:
from collections import defaultdict

ckpt_dirs = sorted(glob.glob(f"{MODELS_DIR}/*_seed*"))
print(f"Found {len(ckpt_dirs)} checkpoint dirs")

ckpts_by_model = defaultdict(list)
for d in ckpt_dirs:
    name = os.path.basename(d)
    key  = "_".join(name.split("_")[:-1])   # strip _seedN
    cp   = os.path.join(d, "best_model.pt")
    if os.path.exists(cp):
        ckpts_by_model[key].append((name, cp))
        print(f"  ✅ {name}")
    else:
        print(f"  ⚠ {name} — best_model.pt missing")


## Inference helper

In [ ]:
@torch.no_grad()
def run_ensemble_on_split(split_df, model_key, model_name, ckpt_list, text_col):
    # Returns averaged binary logits from all seeds of one model.
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    ds = RobustnessDataset(split_df, tokenizer, CONFIG["max_length"], text_col)
    loader = DataLoader(ds, batch_size=CONFIG["batch_size"], shuffle=False,
                        collate_fn=collate_rob, pin_memory=torch.cuda.is_available())

    all_bin_logits = []
    for run_name, ckpt_path in ckpt_list:
        model = MultiTaskTransformer(model_name, TASK_CONFIG).to(device)
        state = torch.load(ckpt_path, map_location=device, weights_only=True)
        model.load_state_dict(state)
        model.eval()

        run_logits = []
        for batch in loader:
            batch = {k: v.to(device, non_blocking=True) for k, v in batch.items()}
            out   = model(batch["input_ids"], batch["attention_mask"],
                          batch.get("token_type_ids"))
            run_logits.append(out["binary"].cpu())

        all_bin_logits.append(torch.cat(run_logits, 0))
        del model; torch.cuda.empty_cache()

    return torch.stack(all_bin_logits, 0).mean(0)   # (N,2) averaged across seeds


def eval_logits(logits, y_true, threshold=0.50):
    probs = F.softmax(logits, -1)[:,1].numpy()
    preds = (probs >= threshold).astype(int)
    valid = y_true >= 0
    return {
        "macro_f1":    round(float(f1_score(y_true[valid], preds[valid], average="macro",    zero_division=0)),4),
        "weighted_f1": round(float(f1_score(y_true[valid], preds[valid], average="weighted", zero_division=0)),4),
        "accuracy":    round(float(accuracy_score(y_true[valid], preds[valid])),4),
        "mcc":         round(float(matthews_corrcoef(y_true[valid], preds[valid])),4),
        "auroc":       round(float(roc_auc_score(y_true[valid], probs[valid])),4),
    }

print("Inference helpers defined ✅")


## Define held-out splits

In [ ]:
ROBUSTNESS_SPLITS = {
    "random_test (in-domain)":          "random_test.csv",
    "source_holdout_banth":             "source_holdout_banth_test.csv",
    "source_holdout_bd_shs":            "source_holdout_bd_shs_test.csv",
    "source_holdout_facebook_44001":    "source_holdout_facebook_44001_test.csv",
    "source_holdout_multilabel_12557":  "source_holdout_multilabel_12557_test.csv",
    "script_holdout_romanized":         "script_holdout_test.csv",
}

print("Split existence check:")
for name, fname in ROBUSTNESS_SPLITS.items():
    p = os.path.join(SPLITS_DIR, fname)
    status = "✅" if os.path.exists(p) else "⚠ MISSING"
    print(f"  {status}  {name}")


## Main robustness loop

In [ ]:
all_rob_results = []

for split_name, split_file in ROBUSTNESS_SPLITS.items():
    split_path = os.path.join(SPLITS_DIR, split_file)
    if not os.path.exists(split_path):
        print(f"⚠ Skipping {split_name} — file not found")
        continue

    split_df = pd.read_csv(split_path)
    # Apply consolidation to this split too
    if "label_type" in split_df.columns:
        split_df["label_type"] = split_df["label_type"].apply(consolidate_type)

    y_true = split_df["label_binary"].values

    print(f"\n{'='*55}")
    print(f"  {split_name}  ({len(split_df):,} samples)")
    print(f"{'='*55}")

    model_logits_list = []
    for model_key, model_name in CONFIG["models"].items():
        if model_key not in ckpts_by_model:
            print(f"  ⚠ No checkpoints for {model_key} — skip")
            continue
        print(f"  Running {model_key} ({len(ckpts_by_model[model_key])} seeds)...")
        lg = run_ensemble_on_split(split_df, model_key, model_name,
                                    ckpts_by_model[model_key], TEXT_COL)
        model_logits_list.append(lg)

    if not model_logits_list:
        print("  ❌ No models ran — skip split")
        continue

    # Average across all models
    ens_logits = torch.stack(model_logits_list, 0).mean(0)
    metrics    = eval_logits(ens_logits, y_true)
    metrics["split"] = split_name
    metrics["n"]     = len(split_df)
    all_rob_results.append(metrics)

    print(f"  macro-F1={metrics['macro_f1']:.4f}  acc={metrics['accuracy']:.4f}  mcc={metrics['mcc']:.4f}")

print(f"\n✅ Robustness evaluation complete — {len(all_rob_results)} splits")


## Save + display results

In [ ]:
if all_rob_results:
    rob_df = pd.DataFrame(all_rob_results)
    cols   = ["split","n","macro_f1","weighted_f1","accuracy","mcc","auroc"]
    rob_df = rob_df[[c for c in cols if c in rob_df.columns]]

    print("\n" + "="*70)
    print("ROBUSTNESS SUMMARY")
    print("="*70)
    print(rob_df.to_string(index=False))

    os.makedirs("../outputs/robustness", exist_ok=True)
    rob_df.to_csv("../outputs/robustness/robustness_results.csv", index=False)
    print("\n✅ Saved → ../outputs/robustness/robustness_results.csv")
else:
    print("⚠ No results collected.")


---
**Next:** `07_ablations_and_analysis.ipynb`